# Notebook 08 — WWF RGB orthomosaics | Musanga V3_1

**Primary objective:** evaluate **native RGB quality and resolution**, and select orthomosaics for **new exhaustive Musanga annotations** across a broad forest regeneration/disturbance gradient. This is **not** a cross-validation experiment with the old UAV sites.

**Workflow:** A Setup → B GCS inventory → C survey map → D RGB native-resolution audit → E small-window visual QC → F diverse annotation site screening.

**Confirmed pilot:** `UTM33S_Plot135_RGB.tif` is EPSG:32733, **10 cm GSD**, 3 bands, 13 × 3 km and **has no overviews**. A 650-pixel preview of the *entire raster* may force huge reads. **Inspect bounded native-pixel crops instead.**

All GCS operations are read-only. No large mosaic downloads, new annotation labels or modifications to existing V3_1 model data.

## A — Environment, authentication and workspace
Install dependencies and connect Colab to Google Drive / GCS. Earth Engine is optional for the RGB-first objective.

In [ ]:
# 00 — Install dependencies
%pip -q install google-cloud-storage geopandas shapely pyogrio rasterio folium earthengine-api geemap fsspec gcsfs

In [ ]:
# 01 — Authenticate Google Cloud, mount Drive and initialize Earth Engine
from google.colab import auth, drive
from pathlib import Path
import os
auth.authenticate_user()
drive.mount('/content/drive', force_remount=False)
EE_PROJECT = os.environ.get('EE_PROJECT', '').strip()  # specify your registered GEE Cloud project if necessary
import ee
try:
    if EE_PROJECT: ee.Initialize(project=EE_PROJECT)
    else: ee.Initialize()
    print('Earth Engine ready')
except Exception as e:
    print('GEE not initialized (not required for LiDAR exploration):', repr(e))
    print('Set EE_PROJECT and rerun this cell to enable satellite comparisons.')
MUSANGA_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
WORK_DIR = MUSANGA_ROOT/'V3_1'/'NB08_WWF_Lidar'
WORK_DIR.mkdir(parents=True,exist_ok=True)
print('Outputs:',WORK_DIR)

## B — WWF dataset inventory
Inspect bucket object sizes, filenames and matched per-plot RGB, LiDAR and CHM coverage. Metadata only.

In [ ]:
# 02 — Bucket inventory (metadata only)
from google.cloud import storage
import pandas as pd, re
from pathlib import PurePosixPath
BUCKET='carbon-maps'
client=storage.Client()
rows=[]
for b in client.list_blobs(BUCKET):
    if b.name.endswith('/'): continue
    m=re.search(r'(UTM\d{1,2}[NS])[_/-]?(?:Plot|plot)[_-]?(\d+)',b.name,re.I)
    plot_id=f'{m.group(1).upper()}_Plot{int(m.group(2))}' if m else None
    rows.append(dict(path=b.name,extension=PurePosixPath(b.name).suffix.lower(),size_GB=(b.size or 0)/1e9,plot_id=plot_id,updated=str(b.updated)))
df=pd.DataFrame(rows)
df.to_csv(WORK_DIR/'carbon_maps_inventory.csv',index=False)
print('Files:',len(df),'GB:',round(df.size_GB.sum(),1))
display(df.groupby('extension').agg(files=('path','size'),GB=('size_GB','sum')).sort_values('GB',ascending=False).head(20))

In [ ]:
# 03 — Match WWF LiDAR products to plot IDs
patterns={
 'RGB':r'^001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif$',
 'CHM':r'^001_Lidar_Rasters/Lidar_2meter/CHM/.*\.tif$',
 'DTM':r'^001_Lidar_Rasters/Lidar_2meter/DTM/.*\.tif$',
 'DSM':r'^001_Lidar_Rasters/Lidar_2meter/DSM/.*\.tif$',
 'LAS':r'^00-raw-data/Plots/.*\.las$',
 'ECW':r'^00-raw-data/Plots/.*\.ecw$'}
products=pd.DataFrame({'plot_id':sorted(df.plot_id.dropna().unique())})
for kind,pattern in patterns.items():
    counts=df.loc[df.path.str.contains(pattern,case=False,regex=True,na=False)].groupby('plot_id').size()
    products[kind+'_n']=products.plot_id.map(counts).fillna(0).astype(int)
display(products.head(20))
print('Plots with RGB:',(products.RGB_n>0).sum())
products.to_csv(WORK_DIR/'wwf_plot_product_inventory.csv',index=False)

## C — WWF survey geography
Load the 216 georeferenced plot footprints and display a reliable offline-first survey map. Esri satellite is optional; OSM 403 errors are avoided.

In [ ]:
# 04 — Download only small plot GIS sidecars; do not read broken Ferry_Lines geometry
import tempfile
import geopandas as gpd
from pathlib import Path
prefix='003_GIS_Layers/Lidar_Plots/'
local=Path(tempfile.mkdtemp(prefix='wwf_gis_'))
for b in client.list_blobs(BUCKET,prefix=prefix):
    if b.name.lower().endswith(('.shp','.shx','.dbf','.prj','.cpg','.qpj')):
        b.download_to_filename(str(local/Path(b.name).name))
shps=list(local.glob('*.shp'))
assert shps, 'No plot shapefile found'
plots=gpd.read_file(shps[0]).to_crs('EPSG:4326')
plots=plots[plots.geometry.notna() & ~plots.geometry.is_empty].copy()
plots['geometry']=plots.geometry.make_valid()
plots['plot_id']=plots.Name.astype(str).str.strip()
plots['utm_zone']=plots.plot_id.str.extract(r'^(UTM\d{1,2}[NS])',expand=False)
plots=plots.merge(products,on='plot_id',how='left',validate='one_to_one')
print('Features:',len(plots),'Unique plots:',plots.plot_id.nunique())
display(plots.groupby('utm_zone').agg(plots=('plot_id','size'),rgb=('RGB_n',lambda x:(x>0).sum())))
plots.to_file(WORK_DIR/'wwf_lidar_plots.gpkg',driver='GPKG')

In [ ]:
# 05 — Robust interactive map (NO OpenStreetMap requests)
# The previous OSM basemap returned HTTP 403 within Colab.
# Esri satellite is optional; the default offline map always works.
import folium
from folium.plugins import Fullscreen, MiniMap
from IPython.display import display
from html import escape
import pandas as pd

# tiles=None ensures ZERO requests to restricted OSM tile servers.
m = folium.Map(location=[-2.5, 23.5], zoom_start=6, tiles=None,
               control_scale=True, prefer_canvas=True)

# Reliable fallback: blank geographic background (no external tile dependency).
folium.TileLayer(
    tiles='data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAusB9Y9UxQAAAABJRU5ErkJggg==',
    attr='Local blank geographic background', name='Offline (always works)',
    overlay=False, show=True, control=True, max_zoom=19
).add_to(m)

# Optional imagery; switch it on only if your Colab browser can access Esri.
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles © Esri', name='Esri World Imagery (optional)',
    overlay=False, show=False, control=True, max_zoom=19
).add_to(m)

colors={'UTM33S':'#9467bd','UTM34S':'#e3771c','UTM34N':'#2ca02c',
        'UTM35S':'#d62728','UTM35N':'#1f77b4','UTM36N':'#8c564b'}

# Plot geometry gives survey context even if ALL basemap servers fail.
foot=folium.FeatureGroup(name='Survey footprints',show=True)
folium.GeoJson(
    plots[['plot_id','geometry']].to_json(),
    style_function=lambda f: {'color':'#555555','weight':1.5,'fillOpacity':0.08},
    tooltip=folium.GeoJsonTooltip(fields=['plot_id'])
).add_to(foot)
foot.add_to(m)

for zone,group in plots.groupby('utm_zone'):
    layer=folium.FeatureGroup(name=f'{zone}: {len(group)} plots',show=True)
    for _,r in group.iterrows():
        pt=r.geometry.representative_point()
        counts={k:int(r[k+'_n']) if pd.notna(r[k+'_n']) else 0
                for k in ('RGB','LAS','CHM','DTM','DSM')}
        label=escape(str(r.plot_id))
        html='<b>'+label+'</b><br>'+'<br>'.join(f'{k}: {v}' for k,v in counts.items())
        col=colors.get(zone,'gray')
        folium.CircleMarker(
            [pt.y,pt.x],radius=5,color=col,weight=2,fill=True,
            fill_color=col if counts['RGB'] else '#ffffff',fill_opacity=.9,
            tooltip=label,popup=folium.Popup(html,max_width=300)
        ).add_to(layer)
    layer.add_to(m)

# Geographic graticule: geographic reference even with the offline background.
graticule=folium.FeatureGroup(name='Latitude / longitude grid',show=True)
for latitude in range(-12,7,2):
    folium.PolyLine([[latitude,12],[latitude,32]],color='#a0a0a0',
                    weight=.6,opacity=.4,tooltip=f'{latitude}° latitude').add_to(graticule)
for longitude in range(12,33,2):
    folium.PolyLine([[-13,longitude],[7,longitude]],color='#a0a0a0',
                    weight=.6,opacity=.4,tooltip=f'{longitude}° longitude').add_to(graticule)
graticule.add_to(m)

w,s,e,n=plots.total_bounds
m.fit_bounds([[s,w],[n,e]])
Fullscreen().add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

output=WORK_DIR/'wwf_lidar_survey_map.html'
m.save(str(output))
print('Offline-capable map:',output)
print('To see satellite imagery: select Esri World Imagery in the layer control.')
display(m)

### C.1 — Optional Musanga UAV ROIs (context only)
Their spatial overlap is **not** required. Run this cell only if you want to see the original UAV sites on the map.

In [ ]:
# 05A — Overlay the real Musanga V3_1 UAV ROI polygons on the WWF map
# Source of truth: V3_1 audited manifest and its ROI.shp (no guessed coordinates).
from pathlib import Path
from shapely.geometry import box
import geopandas as gpd
import rasterio
import folium
import pandas as pd
from IPython.display import display
import json
from html import escape

drive_root=Path('/content/drive/MyDrive')
final_root=next((q for q in [
    drive_root/'Colab Notebooks/Musanga/Final',
    drive_root/'Musanga/Final'] if q.is_dir()), None)
assert final_root, 'Musanga/Final not found. Mount Drive and check V3_1 paths.'
manifest_file=final_root/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert manifest_file.exists(), (
    'Approved V3_1 input manifest missing: '+str(manifest_file)
    +'. Run V3_1 Notebook 00 / Block 10 first.')
manifest=json.loads(manifest_file.read_text())
roi_file=Path(manifest['roi'])
assert roi_file.is_file(), f'ROI shapefile missing: {roi_file}'
uav_roi=gpd.read_file(roi_file)
assert uav_roi.crs is not None, 'ROI.shp has no CRS; refusing to guess.'
uav_roi=uav_roi[uav_roi.geometry.notna() & ~uav_roi.geometry.is_empty].copy()
uav_roi.geometry=uav_roi.geometry.make_valid()
uav_roi=uav_roi[uav_roi.geometry.notna() & ~uav_roi.geometry.is_empty].copy()
assert len(uav_roi), 'No valid UAV ROI geometries.'

# Assign ROI polygons to a site by physical overlap with real mosaic footprints.
# Intersections are measured in the ROI's native metric CRS, not lat/lon degrees.
mosaic_records=[]
for entry in manifest['mosaics']:
    image_file=Path(entry['path'])
    if not image_file.exists():
        print('Mosaic missing; cannot assign this footprint:',image_file)
        continue
    with rasterio.open(image_file) as ds:
        assert ds.crs is not None, f'No CRS: {image_file}'
        g=gpd.GeoSeries([box(*ds.bounds)],crs=ds.crs).to_crs(uav_roi.crs).iloc[0]
        mosaic_records.append(dict(site=str(entry['site']),file=str(entry['file']),geometry=g))
assert mosaic_records, 'No readable UAV mosaics for ROI attribution.'

# Aggregate overlapping mosaics by forest site, avoiding duplicate MOS geometry.
from shapely.ops import unary_union
site_footprints={
    site:unary_union([x['geometry'] for x in mosaic_records if x['site']==site])
    for site in sorted(set(x['site'] for x in mosaic_records))}
roi_rows=[]
for index,row in uav_roi.iterrows():
    g=row.geometry
    overlaps={site:float(g.intersection(foot).area)
              for site,foot in site_footprints.items()}
    best=max(overlaps,key=overlaps.get)
    if overlaps[best]<=0:
        best='Unassigned'
    roi_rows.append(best)
uav_roi['site']=roi_rows
uav_roi['roi_row']=[int(i) for i in range(len(uav_roi))]
print('UAV ROI features:',len(uav_roi))
print('Site assignment:',uav_roi.site.value_counts().to_dict())
if (uav_roi.site=='Unassigned').any():
    print('WARNING: Unassigned ROI polygons are shown separately, not silently removed.')

roi_wgs=uav_roi.to_crs('EPSG:4326')
roi_colors={'Baego':'#ef4444','MOS':'#f59e0b','Yangambi':'#10b981',
            'Yoko':'#ec4899','Unassigned':'#111827'}
for site, group in roi_wgs.groupby('site'):
    layer=folium.FeatureGroup(name=f'UAV ROI — {site} ({len(group)})',show=True)
    color=roi_colors.get(site,'#111827')
    folium.GeoJson(
        group[['site','roi_row','geometry']].to_json(),
        style_function=lambda feat,c=color:{
            'color':c,'weight':3.5,'fillColor':c,'fillOpacity':.16},
        highlight_function=lambda feat:{'weight':5,'fillOpacity':.3},
        tooltip=folium.GeoJsonTooltip(fields=['site','roi_row'],
            aliases=['Musanga site','ROI row'])
    ).add_to(layer)
    # Distinct labels at true ROI interiors
    for _,rr in group.iterrows():
        pt=rr.geometry.representative_point()
        folium.Marker([pt.y,pt.x],
            icon=folium.DivIcon(html=(
                '<div style="font: bold 11px sans-serif;white-space:nowrap;'
                'text-shadow:1px 1px 2px white,-1px -1px 2px white;color:'
                +color+';">'+escape(site)+'</div>'))
        ).add_to(layer)
    layer.add_to(m)

# Drop old layer controls before adding the revised one.
for key, child in list(m._children.items()):
    if isinstance(child,folium.map.LayerControl):
        del m._children[key]
folium.LayerControl(collapsed=False).add_to(m)

# Plot boundaries remain visible against the complete WWF footprint.
allbounds=pd.concat([
    pd.Series([plots.total_bounds[0],roi_wgs.total_bounds[0]]),
    pd.Series([plots.total_bounds[1],roi_wgs.total_bounds[1]]),
    pd.Series([plots.total_bounds[2],roi_wgs.total_bounds[2]]),
    pd.Series([plots.total_bounds[3],roi_wgs.total_bounds[3]])
],axis=1)
m.fit_bounds([[min(plots.total_bounds[1],roi_wgs.total_bounds[1]),
               min(plots.total_bounds[0],roi_wgs.total_bounds[0])],
              [max(plots.total_bounds[3],roi_wgs.total_bounds[3]),
               max(plots.total_bounds[2],roi_wgs.total_bounds[2])]])

uav_roi.to_file(WORK_DIR/'musanga_v3_1_roi_wwf_overlay.gpkg',driver='GPKG')
map_path=WORK_DIR/'wwf_lidar_plus_musanga_uav_roi.html'
m.save(str(map_path))
print('Saved map:',map_path)
print('Saved georeferenced ROI:',WORK_DIR/'musanga_v3_1_roi_wwf_overlay.gpkg')
display(m)

## D — RGB orthomosaic native-resolution audit
Measure pixel size **from raster transforms**, not nominal filenames. Report footprint dimensions, band count, CRS, overviews, block layout and color interpretation. Avoid full-resolution reads.

In [ ]:
# D1 — Inspect a single RGB mosaic's native metadata (cheap remote header reads)
import rasterio
import pandas as pd
from rasterio.crs import CRS
from pyproj import Geod
sample_path='001_Lidar_Rasters/RGB_mosaic/UTM33S_Plot135_RGB.tif'
uri='/vsigs/'+BUCKET+'/'+sample_path
with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif'):
    with rasterio.open(uri) as src:
        print('File:',sample_path)
        print('CRS:',src.crs)
        print('Width × height:',src.width,'×',src.height)
        print('Bands:',src.count, 'Dtypes:',src.dtypes, 'Color interpretation:',[v.name for v in src.colorinterp])
        print('Pixel size (native CRS units):',src.res)
        print('GSD in cm if CRS projected in metres:',tuple(round(100*abs(v),2) for v in src.res) if src.crs and src.crs.is_projected and src.crs.linear_units=='metre' else 'requires CRS unit conversion')
        print('Extent:',src.bounds)
        print('Internal overviews:',src.overviews(1))
        print('Storage layout:',src.profile.get('tiled'), 'Block shapes:',src.block_shapes)
        print('Compression:',src.profile.get('compress'),'Nodata:',src.nodata)
        print('Pixel count:',f'{src.width*src.height:,}')
print('NOTE: Missing overviews — NEVER downsample entire full-resolution TIFF for an overview.')

### D.2 — Metadata audit across multiple RGB mosaics
Start with **6 mosaics from different UTM zones** (modifiable), not all 216. Remote metadata access can still incur GCS requests. This saves a QC table with resolution in centimetres and flags missing overviews.

In [ ]:
# D2 — Sampled metadata audit: no image pixel reads
import math, pandas as pd, rasterio
MAX_RGB_METADATA=6
# One example per UTM zone; choose more later when working
rgb_paths=(df[df.path.str.fullmatch(r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\\.tif',case=False,na=False)]
           .copy())
rgb_paths['utm']=rgb_paths.plot_id.str.extract(r'^(UTM\\d{1,2}[NS])',expand=False)
pick=rgb_paths.groupby('utm',sort=True).head(1).head(MAX_RGB_METADATA)
metadata_rows=[]
for rec in pick.itertuples(index=False):
    row=dict(plot_id=rec.plot_id,path=rec.path,size_GB=rec.size_GB)
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open('/vsigs/'+BUCKET+'/'+rec.path) as s:
                row.update(crs=str(s.crs),width=s.width,height=s.height,
                           bands=s.count,gsd_x_native=s.res[0],gsd_y_native=s.res[1],
                           overviews=len(s.overviews(1)),tiled=s.profile.get('tiled'),
                           compression=s.profile.get('compress'),
                           block_x=s.block_shapes[0][1],block_y=s.block_shapes[0][0])
                if s.crs and s.crs.is_projected and s.crs.linear_units=='metre':
                    row.update(gsd_cm_x=100*abs(s.res[0]),gsd_cm_y=100*abs(s.res[1]))
                else:
                    row.update(gsd_cm_x=float('nan'),gsd_cm_y=float('nan'))
    except Exception as e:
        row['error']=str(e)[:250]
    metadata_rows.append(row)
meta=pd.DataFrame(metadata_rows)
meta.to_csv(WORK_DIR/'wwf_rgb_metadata_audit.csv',index=False)
display(meta)
print('Saved:',WORK_DIR/'wwf_rgb_metadata_audit.csv')

## E — RGB quality inspection at **native resolution**
The main annotation decision requires individual tree crown visibility. **Do not use `src.read(out_shape=(3,650,650))` on an entire 13 km mosaic without overviews.** Instead, read **small native-pixel windows** (40 × 40 m at 10 cm = 400 × 400 pixels). Inspect a few different locations within each mosaic; the default is 4 small windows on the first pilot mosaic.

Note: a large striped TIFF can cause some extra data transfer even for a small window; inspect block shapes and monitor network usage.

In [ ]:
# E1 — 4 native-resolution image windows, no full mosaic download
import numpy as np, matplotlib.pyplot as plt, rasterio
from rasterio.windows import Window
from pathlib import Path
QC_PLOT_ID='UTM33S_Plot135'
QC_WINDOW_METRES=40
QC_MAX_WINDOWS=4
QC_FRACTIONS=[(0.25,0.25),(0.75,0.25),(0.25,0.75),(0.75,0.75)]
source_rows=df[df.path.str.fullmatch(
    r'001_Lidar_Rasters/RGB_mosaic/'+QC_PLOT_ID+r'_RGB\\.tif',case=False,na=False)]
assert len(source_rows)==1,f'Expected one RGB mosaic for {QC_PLOT_ID}, found {len(source_rows)}'
qc_path=source_rows.iloc[0]['path']
preview_dir=WORK_DIR/'RGB_QC_native_windows'
preview_dir.mkdir(parents=True,exist_ok=True)
with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
    with rasterio.open('/vsigs/'+BUCKET+'/'+qc_path) as src:
        assert src.crs and src.crs.is_projected and src.crs.linear_units=='metre'
        assert src.count>=3, 'Expected at least 3 RGB bands'
        cw=max(1,round(QC_WINDOW_METRES/abs(src.res[0])))
        ch=max(1,round(QC_WINDOW_METRES/abs(src.res[1])))
        cw=min(cw,src.width);ch=min(ch,src.height)
        print('Native resolution:',src.res,'m | window:',cw,'×',ch,'px')
        print('Raster block layout:',src.block_shapes,'| overviews:',src.overviews(1))
        for k,(fx,fy) in enumerate(QC_FRACTIONS[:QC_MAX_WINDOWS],1):
            col=max(0,min(src.width-cw,round(fx*src.width-cw/2)))
            row=max(0,min(src.height-ch,round(fy*src.height-ch/2)))
            win=Window(col,row,cw,ch)
            # native image window ONLY: no raster-wide resampling
            a=src.read(indexes=[1,2,3],window=win,masked=True).astype('float32')
            a=np.ma.filled(a,np.nan)
            img=np.zeros((ch,cw,3),dtype=np.float32)
            for b in range(3):
                z=a[b];ok=np.isfinite(z)
                if ok.any():
                    lo,hi=np.percentile(z[ok],[2,98])
                    img[:,:,b]=np.clip((z-lo)/max(hi-lo,1e-6),0,1)
            fig,ax=plt.subplots(figsize=(7,7))
            ax.imshow(img,interpolation='nearest')
            ax.set_title(f'{QC_PLOT_ID} — native {src.res[0]*100:.1f} cm — window {k}')
            ax.axis('off')
            out=preview_dir/f'{QC_PLOT_ID}_native_window{k}.png'
            fig.savefig(out,dpi=160,bbox_inches='tight')
            plt.show();plt.close(fig)
            print('Saved:',out)
print('Inspect texture, sharpness, color fidelity, crown separation and mosaic seams.')

### E.2 — Manual visual quality score
Record whether crowns can be delineated, not merely nominal pixel size. Evaluate blur, shadows, mosaicking seams, color balance and the visibility of small pioneer crowns. Save a simple screening table for each inspected sample.

In [ ]:
# E2 — Editable QC sheet (fill after examining true-resolution windows)
qc_form=pd.DataFrame([{
    'plot_id':'UTM33S_Plot135','gsd_cm':10.0,
    'crown_visibility_1to5':None,'sharpness_1to5':None,
    'shadow_problem_1to5':None,'seam_problem_1to5':None,
    'possible_musanga_visible':'UNCERTAIN',
    'forest_structure':'UNREVIEWED',
    'exhaustive_annotation_feasible':'UNREVIEWED',
    'notes':'Manual review required; no species inference from GSD alone'
}])
qc_form.to_csv(WORK_DIR/'wwf_rgb_visual_qc_template.csv',index=False)
display(qc_form)
print('QC template:',WORK_DIR/'wwf_rgb_visual_qc_template.csv')

## F — Screening mosaics for new exhaustive annotations
Choose spatially dispersed candidate RGB mosaics as a starting list. **Spatial dispersion is not a proxy for forest age.** Classification into young secondary forest, regrowth, disturbed forest, mature canopy or open land requires image review and external land-cover history.

In [ ]:
# 08 — Catalogue RGB mosaics and select a spatially dispersed screening set
import numpy as np, pandas as pd
from pathlib import PurePosixPath
rgb=df[df.path.str.fullmatch(r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif',case=False,na=False)].copy()
geo=plots.copy()
cent=geo.geometry.representative_point()
geo['longitude']=cent.x;geo['latitude']=cent.y
catalog=geo[['plot_id','utm_zone','longitude','latitude','geometry']].drop(columns='geometry').merge(
    rgb[['plot_id','path','size_GB']].rename(columns={'path':'rgb_gcs_path','size_GB':'rgb_GB'}),
    how='inner',on='plot_id',validate='one_to_one')
catalog['visual_forest_class']='UNREVIEWED'
catalog['musanga_presence']='UNREVIEWED'
catalog['annotation_suitability']='UNREVIEWED'
catalog['notes']=''
catalog.to_csv(WORK_DIR/'wwf_rgb_annotation_catalog.csv',index=False)
print('RGB candidates:',len(catalog))
display(catalog.groupby('utm_zone').agg(n=('plot_id','size'),GB=('rgb_GB','sum')))
# Screen a geographically dispersed subset across all UTM zones.
# Farthest-point sampling on geographic coordinates within zones; not interpreted as age.
from pyproj import Geod
geod=Geod(ellps='WGS84')
chosen=[]
for zone, group in catalog.groupby('utm_zone'):
    target=min(5,len(group))
    pool=group.reset_index(drop=True)
    selected=[0]
    while len(selected)<target:
        distances=[]
        for j,rr in pool.iterrows():
            if j in selected: distances.append(-1);continue
            mind=min(geod.inv(rr.longitude,rr.latitude,
                              pool.iloc[k].longitude,pool.iloc[k].latitude)[2]
                     for k in selected)
            distances.append(mind)
        selected.append(int(np.argmax(distances)))
    chosen.append(pool.iloc[selected])
screen=pd.concat(chosen,ignore_index=True)
screen['review_priority']='INITIAL_SCREEN'
screen.to_csv(WORK_DIR/'wwf_rgb_initial_screen.csv',index=False)
print('Initial screening set:',len(screen),'(spatially dispersed, NOT preclassified)')
display(screen[['plot_id','utm_zone','latitude','longitude','rgb_GB']])

## G — Next decisions
Compare native GSD and observed RGB quality across zones; shortlist mosaics with visually annotatable crowns and a balanced gradient of forest environments, including Musanga-free exhaustive negatives. Expand the screening set only after reviewing the pilot true-resolution crops.